# 02 — Statistical Analysis
## "Your Zip Code is Your Health Sentence"

Runs all three analysis phases:
- **Phase 2:** Food Access → Chronic Disease (OLS, odds ratios, partial correlations)
- **Phase 3:** The Zip Code Effect (variance decomposition, life expectancy regression)
- **Phase 4:** Race as a Residual Gap (interaction terms, residual analysis)
- **Phase 5B:** Health Disadvantage Index (weighted composite, tract ranking)

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

from src.analysis.health_stats import run_phase2, run_phase3, run_phase4
from src.analysis.health_index import build_health_disadvantage_index

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 6)})
%matplotlib inline

DATA_PROCESSED = Path("..") / "data" / "processed"

In [ ]:
# Load master dataset
df = pd.read_parquet(DATA_PROCESSED / "master.parquet")
print(f"Master loaded: {df.shape[0]:,} rows x {df.shape[1]} cols")
print(f"Usable rows (has food desert + diabetes): {df.dropna(subset=['is_food_desert', 'diabetes_pct']).shape[0]:,}")

## Phase 2: Food Access → Chronic Disease

In [ ]:
p2 = run_phase2(df)

# Display key findings
if "odds_ratio_diabetes" in p2:
    od = p2["odds_ratio_diabetes"]
    print(f"\n{'='*60}")
    print(f"KEY FINDING: Food desert residents have {od['odds_ratio']:.2f}x the odds")
    print(f"of above-median diabetes prevalence (95% CI: {od['ci_95_low']:.2f}–{od['ci_95_high']:.2f})")
    print(f"  Desert mean diabetes: {od['desert_mean_diabetes']}%")
    print(f"  Non-desert mean:      {od['non_desert_mean_diabetes']}%")
    print(f"{'='*60}")

if "ols_diabetes" in p2:
    print(f"\nOLS R² (diabetes): {p2['ols_diabetes']['r_squared']}")
    print("Coefficients:")
    for var, vals in p2["ols_diabetes"]["coefficients"].items():
        sig = "***" if vals["p_value"] < 0.001 else "**" if vals["p_value"] < 0.01 else "*" if vals["p_value"] < 0.05 else ""
        print(f"  {var}: {vals['coef']:.4f} (p={vals['p_value']}) {sig}")

## Phase 3: The Zip Code Effect

In [ ]:
p3 = run_phase3(df)

if "standardized_betas" in p3:
    betas = p3["standardized_betas"]
    print("\nStandardized Betas (Life Expectancy):")
    sorted_betas = sorted(betas.items(), key=lambda x: abs(x[1]), reverse=True)
    for var, beta in sorted_betas:
        bar = "█" * int(abs(beta) * 30)
        direction = "+" if beta > 0 else "-"
        print(f"  {var:30s} β={beta:+.3f} {direction}{bar}")

if "life_expectancy_gap" in p3:
    gap = p3["life_expectancy_gap"]
    print(f"\n{'='*60}")
    print(f"KEY FINDING: Life expectancy gap between richest and poorest")
    print(f"income quintiles: {gap['gap_years']:.1f} years")
    print(f"  Poorest quintile: {gap['poorest_quintile_mean_le']} years")
    print(f"  Richest quintile: {gap['richest_quintile_mean_le']} years")
    print(f"{'='*60}")

## Phase 4: Race as a Residual Gap

In [ ]:
p4 = run_phase4(df)

if "residual_analysis" in p4:
    ra = p4["residual_analysis"]
    print(f"\n{'='*60}")
    print(f"RESIDUAL ANALYSIS: Is race significant after controlling for income + food access?")
    print(f"  R² without race: {ra['r2_without_race']}")
    print(f"  R² with race:    {ra['r2_with_race']}")
    print(f"  ΔR²:             {ra['r2_change']}")
    print(f"  pct_black coef:  {ra['pct_black_coefficient']} (p={ra['pct_black_p_value']})")
    print(f"  Race significant: {'YES' if ra['race_still_significant'] else 'NO'}")
    print(f"{'='*60}")

if "cross_comparison" in p4:
    cc = p4["cross_comparison"]
    print(f"\nCROSS-COMPARISON:")
    print(f"  High-income Black tracts: {cc['high_income_black_mean_diabetes']}% diabetes (n={cc['n_high_income_black']})")
    print(f"  Low-income White tracts:  {cc['low_income_white_mean_diabetes']}% diabetes (n={cc['n_low_income_white']})")
    print(f"  → Income {'DOES NOT close' if not cc['income_closes_gap'] else 'closes'} the racial gap")

## Phase 5B: Health Disadvantage Index

In [ ]:
p5 = build_health_disadvantage_index(df, phase2_results=p2, phase3_results=p3)

if "decile_gaps" in p5:
    print(f"\n{'='*60}")
    print("HEALTH DISADVANTAGE INDEX — Decile Gaps")
    print(f"Tracts scored: {p5['n_tracts_scored']:,}")
    for metric, vals in p5["decile_gaps"].items():
        print(f"\n  {metric}:")
        print(f"    Most disadvantaged 10%:  {vals['most_disadvantaged']}")
        print(f"    Least disadvantaged 10%: {vals['least_disadvantaged']}")
        print(f"    Gap: {vals['gap']}")
    print(f"{'='*60}")

if "path_diagram" in p5 and p5["path_diagram"]:
    print("\nPATH DIAGRAM COEFFICIENTS:")
    for path, vals in p5["path_diagram"].items():
        print(f"  {path}: β={vals['coef']}, R²={vals['r_squared']}")